# 05 Export + Inference: ส่งออกโมเดลและรันตรวจจับลง CSV (Phase 5)

- kernel **`Python (rwd)`**, ต้องมีโมเดลที่เทรนจบจาก `02_train.ipynb` ไม่ต้องใช้ GPU (ทุกอย่างในไฟล์นี้รันบน CPU)
- โมเดลที่ใช้: `yolo26n-seg` รอบแรก (ตัวที่มีผล test) เปลี่ยนได้ที่ `CONFIG`
- **ตรวจจับอย่างเดียว**: ผลลัพธ์คือไฟล์ CSV และภาพซ้อน **ห้ามต่อกับหัวฉีด โซลินอยด์ หรืออุปกรณ์สั่งงานใด ๆ**
- ความเร็วที่วัดได้คือ **CPU ของเครื่องนี้เท่านั้น** ไม่ใช่ผลบน Raspberry Pi หรือบนโดรน
- ไฟล์นี้ใช้ **val** เท่านั้น ไม่แตะ test ตัวเลขความแม่นบน val ห้ามนำไปเผยแพร่
- รันซ้ำได้: ขั้นที่ทำไปแล้วจะอ่านจากไฟล์ ตั้ง `RERUN = True` เพื่อทำใหม่

## 1. Setup

In [ ]:
import os, sys, json, csv
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
PY = sys.executable
if ".venv" not in PY:
    print("WARNING: kernel is not 'Python (rwd)'. Run scripts/setup_env.sh, then Kernel -> Change Kernel.")

from rwd.train import load_train_config
CONFIG = ROOT / "configs/train.yaml"      # โมเดลที่เลือก
cfg = load_train_config(CONFIG)
RUN = Path(cfg["project"]) / cfg["name"]
W = RUN / "weights"
DATA = Path(cfg["data"])
VAL_IMAGES = DATA.parent / "images/val"
CONF = 0.25                                # ค่าเดียวกับที่ใช้วัดบน test
RERUN = False
print("run:", RUN.name, "| best.pt exists:", (W / "best.pt").exists())

In [ ]:
# แพ็กเกจสำหรับ export (onnx, onnxruntime, ncnn) ลง .venv เท่านั้น
!"{PY}" -m pip install -q -r requirements-export.txt

## 2. Export เป็น ONNX และ NCNN

ขนาด input คงที่ 960x1280 (สูง x กว้าง) เท่าภาพในชุดข้อมูล ไฟล์อยู่ในโฟลเดอร์ `weights/` ของ run (ไม่เข้า git)

In [ ]:
if RERUN or not (RUN / "export.json").exists():
    !"{PY}" -m rwd.export --weights "{W / 'best.pt'}"
for r in json.loads((RUN / "export.json").read_text())["exports"]:
    print(f"{r['format']:<5} {r['size_mb']:>7} MB   {Path(r['path']).name}")

## 3. เช็คว่าโมเดลที่ export ให้ผลเหมือนต้นฉบับ (บน val)

ผล test วัดจากไฟล์ `.pt` เท่านั้น ขั้นนี้ดูว่า ONNX และ NCNN ให้ค่าระดับพิกเซลบน val ตรงกับ `.pt` ที่ conf เดียวกันไหม
ถ้าต่างกันมากแปลว่า export มีปัญหา **ตัวเลข val ห้ามนำไปเผยแพร่**

In [ ]:
for name, tag in (("best.onnx", "onnx"), ("best_ncnn_model", "ncnn")):
    if RERUN or not (RUN / f"eval_val_{tag}.json").exists():
        !"{PY}" -m rwd.evaluate --weights "{W / name}" --data "{DATA}" --split val --conf {CONF} --imgsz 960 1280 --device cpu --tag {tag} > /dev/null 2>&1
key = f"{CONF:g}"
print("[val: ใช้เช็ค export เท่านั้น ไม่ใช่ผลที่เผยแพร่ได้]  conf", key)
print(f"{'model':<8} {'precision':>10} {'recall':>8} {'IoU':>8}")
ref = None
for label, f in (("pt", "eval_val.json"), ("onnx", "eval_val_onnx.json"), ("ncnn", "eval_val_ncnn.json")):
    a = json.loads((RUN / f).read_text())["pixel"][key]["all"]
    ref = ref or a
    print(f"{label:<8} {a['precision']:>10.4f} {a['recall']:>8.4f} {a['iou']:>8.4f}   ต่างจาก pt (IoU): {a['iou'] - ref['iou']:+.4f}")

## 4. ความเร็วบน CPU ของเครื่องนี้

วัดทีละรูปแบบใน process แยก 20 ภาพจาก val หลัง warm-up 3 ภาพ รวมเวลาเตรียมภาพและประมวลผล mask
**เป็นค่าประมาณคร่าว ๆ ของเครื่องนี้ ไม่ใช่ผลบน Raspberry Pi หรือบอร์ดอื่น**

In [ ]:
BENCH = RUN / "benchmark_cpu.json"
if RERUN or not BENCH.exists():
    !"{PY}" -m rwd.benchmark --weights "{W / 'best.pt'}" "{W / 'best.onnx'}" "{W / 'best_ncnn_model'}" --images "{VAL_IMAGES}" --n 20 --conf {CONF} --out "{BENCH}" > /dev/null
b = json.loads(BENCH.read_text())
print("CPU:", b["cpu"], f"({b['logical_cores']} logical cores)")
print(f"{'model':<18} {'median ms':>10} {'p90 ms':>8} {'ภาพ/วินาที':>11} {'RAM สูงสุด MB':>14}")
for r in b["results"]:
    print(f"{r['weights']:<18} {r['median_ms']:>10} {r['p90_ms']:>8} {r['fps_median']:>11} {r['peak_rss_mb']:>14}")

## 5. รันตรวจจับแล้วบันทึกลง CSV

- `SOURCE`: โฟลเดอร์ภาพ (หรือไฟล์ภาพเดียว) เปลี่ยนเป็นโฟลเดอร์ภาพของคุณเองได้
- หนึ่งแถวต่อหนึ่งบริเวณที่เจอ ภาพที่ไม่เจออะไรก็มีหนึ่งแถว (`detection = 0`)
- พิกัดและความสูงอ่านจากชื่อไฟล์ ถ้าชื่อมีรูป `lat_.._lon_.._alt_..m` ไม่มีก็เว้นว่าง
- CSV และภาพซ้อนอยู่ใต้ `runs/` ไม่เข้า git

In [ ]:
SOURCE = VAL_IMAGES
MODEL = W / "best.onnx"          # หรือ best.pt, best_ncnn_model
LIMIT = 12                       # None = ทุกภาพในโฟลเดอร์
OUT_CSV = RUN / "detections_demo.csv"
OVERLAYS = RUN / "detections_demo_overlays"
limit = f"--limit {LIMIT}" if LIMIT else ""
!"{PY}" -m rwd.infer --weights "{MODEL}" --source "{SOURCE}" --conf {CONF} --out "{OUT_CSV}" --overlays "{OVERLAYS}" {limit}

In [ ]:
rows = list(csv.DictReader(open(OUT_CSV)))
cols = ["image", "detection", "class", "confidence", "mask_area_px", "lat", "lon", "alt_m"]
print("  ".join(f"{c:>12}" for c in cols))
for r in rows[:12]:
    print("  ".join(f"{(r[c][-12:] if c == 'image' else r[c]):>12}" for c in cols))
print(f"... {len(rows)} แถว")

# สรุปต่อภาพ: จำนวนบริเวณ และสัดส่วนพื้นที่ภาพที่โมเดลทายว่าเป็น weedy_rice
print(f"\n{'image':<30} {'บริเวณ':>7} {'% พื้นที่ภาพ':>12}")
per = {}
for r in rows:
    p = per.setdefault(r["image"], [0, 0, int(r["image_width"]) * int(r["image_height"])])
    if r["detection"] != "0":
        p[0] += 1
        p[1] += int(r["mask_area_px"] or 0)
for name, (n, area, total) in per.items():
    print(f"{name[13:40]:<30} {n:>7} {100 * area / total:>11.1f}%")

In [ ]:
import cv2
import matplotlib.pyplot as plt

files = sorted(OVERLAYS.glob("*.jpg"))[:3]
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for ax, p in zip(axes, files):
    ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)); ax.set_title(p.name[13:40], fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## Next

- บนเครื่องอื่น (เช่น MacBook): คัดลอก `best.onnx` ไป แล้วรัน `python -m rwd.benchmark` และ `python -m rwd.infer` แบบเดียวกัน
- รายการตรวจสำหรับ Raspberry Pi 5 อยู่ที่ `docs/EDGE_CHECKLIST.md` **ยังไม่เคยทดสอบบนบอร์ดจริง**
- ดูสรุปทั้งโปรเจกต์ที่ `00_overview.ipynb`